与 `quantstats` 量化分析库之间的核心适配器模块。该模块通过智能的适配器模式，将 quantstats 库的所有功能无缝集成到 vectorbt 的收益率分析体系中。

quantStats 是一个专门用于投资组合分析的 Python 库，由三个主要模块组成
- stats：计算各种绩效指标，如夏普比率、胜率、波动率等
- plots：可视化绩效、回撤、滚动统计、月度收益等
- reports：生成指标报告、批量绘图和可保存为 HTML 文件的分析报告

# 类继承关系
```mermaid
classDiagram
    class Configured
    class QSAdapter
    class ReturnsAccessor

    %% 继承关系
    Configured <|-- QSAdapter
    QSAdapter o-- ReturnsAccessor : has
```

# attach_qs_methods

用于装饰类，例如：
```python
@attach_qs_methods
class QSAdapter(Configured): ...
```
被装饰后的类 `QSAdapter` 拥有 quantstats 库的所有功能。

## 源码

```python
def attach_qs_methods(cls: tp.Type[tp.T], replace_signature: bool = True) -> tp.Type[tp.T]:
    checks.assert_subclass_of(cls, "QSAdapter")

    for module_name in ['utils', 'stats', 'plots', 'reports']:
        for qs_func_name, qs_func in getmembers(getattr(qs, module_name), isfunction):
            if not qs_func_name.startswith('_') and checks.func_accepts_arg(qs_func, 'returns'):
                if module_name == 'plots':
                    new_method_name = 'plot_' + qs_func_name
                elif module_name == 'reports':
                    new_method_name = qs_func_name + '_report'
                else:
                    new_method_name = qs_func_name

                def new_method(self, *, _func: tp.Callable = qs_func, **kwargs) -> tp.Any:
                    returns = self.returns_accessor.obj
                    if isinstance(returns, pd.DataFrame):
                        null_mask = returns.isnull().any(axis=1)
                    else:
                        null_mask = returns.isnull()
                    func_arg_names = get_func_arg_names(_func)
                    defaults = self.defaults

                    pass_kwargs = dict()
                    for arg_name in func_arg_names:
                        if arg_name not in kwargs:
                            if arg_name in defaults:
                                pass_kwargs[arg_name] = defaults[arg_name]
                            elif arg_name == 'benchmark':
                                if self.returns_accessor.benchmark_rets is not None:
                                    pass_kwargs['benchmark'] = self.returns_accessor.benchmark_rets
                            elif arg_name == 'periods':
                                pass_kwargs['periods'] = int(self.returns_accessor.ann_factor)
                            elif arg_name == 'periods_per_year':
                                pass_kwargs['periods_per_year'] = int(self.returns_accessor.ann_factor)
                        else:
                            pass_kwargs[arg_name] = kwargs[arg_name]

                    if 'benchmark' in pass_kwargs:
                        if isinstance(pass_kwargs['benchmark'], pd.DataFrame):
                            bm_null_mask = pass_kwargs['benchmark'].isnull().any(axis=1)
                        else:
                            bm_null_mask = pass_kwargs['benchmark'].isnull()
                        null_mask = null_mask | bm_null_mask
                        pass_kwargs['benchmark'] = pass_kwargs['benchmark'].loc[~null_mask]
                    returns = returns.loc[~null_mask]

                    signature(_func).bind(returns=returns, **pass_kwargs)
                    return _func(returns=returns, **pass_kwargs)

                if replace_signature:
                    # Replace the function's signature with the original one
                    source_sig = signature(qs_func)
                    new_method_params = tuple(signature(new_method).parameters.values())
                    self_arg = new_method_params[0]
                    other_args = [
                        p.replace(kind=Parameter.KEYWORD_ONLY)
                        if p.kind in (Parameter.POSITIONAL_ONLY, Parameter.POSITIONAL_OR_KEYWORD)
                        else p
                        for p in list(source_sig.parameters.values())[1:]
                    ]
                    source_sig = source_sig.replace(parameters=(self_arg,) + tuple(other_args))
                    new_method.__signature__ = source_sig

                new_method.__doc__ = f"See `quantstats.{module_name}.{qs_func_name}`."
                new_method.__qualname__ = f"{cls.__name__}.{new_method_name}"
                new_method.__name__ = new_method_name
                setattr(cls, new_method_name, new_method)
    return cls
```

# class QSAdapter(Configured)
注意：
```python
@attach_qs_methods
class QSAdapter(Configured): ...
```
另外，类 `ReturnsAccessor` 中有方法
```python
@property
def qs(self):
    from vectorbt.returns.qs_adapter import QSAdapter
    return QSAdapter(self)
```
所以，对于一个 `ReturnsAccessor` 类型的实例 `x`，`x.qs` 就创建了一个 `QSAdapter(x)` 实例，该实例可以调用 quantstats 库中的所有方法。

## `__init__`
```python
    def __init__(self, returns_accessor: ReturnsAccessor, defaults: tp.KwargsLike = None, **kwargs) -> None:
        checks.assert_instance_of(returns_accessor, ReturnsAccessor)

        Configured.__init__(self, returns_accessor=returns_accessor, defaults=defaults, **kwargs)

        self._returns_accessor = returns_accessor
        self._defaults = defaults
```

## 例子

In [1]:
import numpy as np
import pandas as pd
import vectorbt as vbt
import quantstats as qs
import matplotlib.pyplot as plt

# 第一步：准备测试数据
# 创建模拟的收益率数据
np.random.seed(42)
dates = pd.date_range('2020-01-01', '2023-12-31', freq='D')
returns = pd.Series(
    np.random.normal(0.0008, 0.02, len(dates)),  # 日收益率：均值0.08%，标准差2%
    index=dates,
    name='策略收益率'
)

# 创建基准收益率（模拟市场指数）
benchmark_returns = pd.Series(
    np.random.normal(0.0005, 0.015, len(dates)),  # 基准：均值0.05%，标准差1.5%
    index=dates,
    name='基准收益率'
)

print("数据准备完成:")
print(f"收益率数据形状: {returns.shape}")
print(f"时间范围: {returns.index[0]} 到 {returns.index[-1]}")
print(f"收益率统计: 均值={returns.mean():.6f}, 标准差={returns.std():.6f}")

数据准备完成:
收益率数据形状: (1461,)
时间范围: 2020-01-01 00:00:00 到 2023-12-31 00:00:00
收益率统计: 均值=0.001700, 标准差=0.019755


In [2]:
# 第二步：创建 ReturnsAccessor 并配置参数
ret_accessor = returns.vbt.returns(
    benchmark_rets=benchmark_returns,  # 设置基准收益率
    freq='D',                          # 数据频率：日频
    year_freq='252D',                  # 年化频率：252个交易日
    defaults=dict(
        risk_free=0.02,               # 无风险利率：2%
        periods=252,                  # 年化周期数
        periods_per_year=252          # 每年周期数
    )
)

In [5]:
# 第三步：获取 QSAdapter 实例（已通过 @attach_qs_methods 装饰器增强）
qs_adapter = ret_accessor.qs

# 检查所有可用的方法
print("=== QSAdapter 可用方法列表 ===")
available_methods = [method for method in dir(qs_adapter) if not method.startswith('_')]
print(f"总共有 {len(available_methods)} 个方法")

# 查找与 drawdown 相关的方法
drawdown_methods = [method for method in available_methods if 'drawdown' in method.lower()]
print(f"\n与 drawdown 相关的方法: {drawdown_methods}")

# 查找统计相关的方法
stats_methods = [method for method in available_methods if any(keyword in method.lower() 
                 for keyword in ['sharpe', 'calmar', 'sortino', 'var', 'volatility'])]
print(f"\n统计相关的方法: {stats_methods}")

=== QSAdapter 可用方法列表 ===
总共有 110 个方法

与 drawdown 相关的方法: ['plot_drawdown', 'plot_drawdowns_periods', 'to_drawdown_series']

统计相关的方法: ['adjusted_sortino', 'calmar', 'cvar', 'implied_volatility', 'plot_rolling_sharpe', 'plot_rolling_sortino', 'plot_rolling_volatility', 'rolling_sharpe', 'rolling_sortino', 'rolling_volatility', 'sharpe', 'smart_sharpe', 'smart_sortino', 'sortino', 'var', 'volatility']


In [6]:
# 第四步：使用来自 quantstats.stats 模块的方法
print("\n=== 统计分析方法示例 ===")

# 这些方法都是通过 attach_qs_methods 装饰器动态添加的
performance_metrics = {
    '夏普比率': qs_adapter.sharpe(),
    '年化收益率': qs_adapter.cagr(),
    '波动率': qs_adapter.volatility(),
    '卡尔玛比率': qs_adapter.calmar(),
    '索提诺比率': qs_adapter.sortino(),
    'VaR(95%)': qs_adapter.var(confidence=0.95),
    '胜率': qs_adapter.win_rate(),
    '盈亏比': qs_adapter.payoff_ratio()
}

for metric_name, value in performance_metrics.items():
    print(f"{metric_name}: {value:.4f}")


=== 统计分析方法示例 ===
夏普比率: 1.3033
年化收益率: 0.4615
波动率: 0.3136
卡尔玛比率: 1.2668
索提诺比率: 1.9867
VaR(95%): -0.0308
胜率: 0.5421
盈亏比: 1.0488
